🟢 Débutant | ⏱ 45 min | 🔑 ton module, réponse d'API, fusion

# Module 09 — Fil rouge MétéoLocale : prévisions et historique

**Lundi matin chez MétéoLocale.** Inès arrive avec son café : « Le site veut afficher, pour Toulouse, septembre
**et** les prévisions de la semaine. Karim a appelé l'API Open-Meteo ce matin et a enregistré la réponse dans
`open_meteo_toulouse.json`. Et s'il te plaît, arrête de recopier ta boîte à outils dans chaque notebook : range-la
dans un **module** `meteolocale.py`. »

**Ce qu'on construit aujourd'hui** : ton module `sorties/meteolocale.py`, puis la fusion historique + prévisions,
quelques statistiques et un fichier JSON pour le site.

**Ce qu'on a construit avant** :
- 01 à 05 : une journée mise en forme et classée, puis les 30 jours de Toulouse dans une liste ;
- 06 : le jeu de données en liste de dictionnaires ;
- 07 : la boîte à outils de fonctions (`moyenne`, `celsius_vers_fahrenheit`, `resume_ville`) ;
- 08 : le vrai `meteo.csv` chargé, `meteo_brut.csv` nettoyé, le résumé de septembre en JSON.

Les étapes s'enchaînent : exécute-les dans l'ordre.

> 🌐 Pourquoi pas l'API en direct ? Parce que la réponse change chaque jour (et qu'il faut du réseau) : avec le
> fichier figé, tout le monde obtient les mêmes résultats. Dans un vrai projet, l'étape 3 commencerait par
> `requests.get(...)`, exactement comme dans le cours (section 9).

### Lire un échec de vérification

Quand une vérification échoue, Python affiche un long message rouge, surtout en anglais. **Ne lis que la
dernière ligne** :

```
AssertionError: moyenne doit valoir 21.5, tu as obtenu 25.0 : as-tu divisé par le nombre de jours ?
```

- `AssertionError` veut dire « **la vérification a échoué** » (ton code tourne, mais le résultat n'est pas encore le bon) ;
- le **message en français**, après les deux-points, te dit **quoi corriger** ;
- tout ce qui est au-dessus (en anglais, avec des numéros de ligne) peut être **ignoré**.

Si la dernière ligne commence par un autre mot (`NameError`, `TypeError`...), c'est **ta cellule** qui a un
problème : relis la section ⚠️ du cours qui parle de cette erreur.

Les caractères spéciaux (émojis comme ☀️ 🌧️ 💨, flèche →, symbole °) ne se tapent pas facilement : **copie-colle-les
depuis l'énoncé**.

In [ ]:
import importlib
import json
import statistics
from datetime import date
from pathlib import Path

Path("sorties").mkdir(exist_ok=True)
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
print("Prêt.")

### Étape 1 — Ton module `meteolocale.py`

**Objectif** : Ranger ta boîte à outils dans un module et l'importer.

**À faire** :
1. Complète, **dans le texte `code`**, les deux `...` : la formule de `moyenne` et celle de `celsius_vers_fahrenheit`.
2. La cellule écrit `sorties/meteolocale.py`, l'importe et le recharge.

**Résultat attendu** :
```
Module meteolocale prêt : 6 villes
```

<details><summary>💡 Indice</summary>

Les deux formules du module 07 : `sum(valeurs) / len(valeurs)` et `round(celsius * 9 / 5 + 32, chiffres)`. Si tu corriges et relances, la ligne `importlib.reload` garantit que la nouvelle version est lue.

📖 Cours, section 6.

</details>

In [ ]:
code = """
# meteolocale.py : la boîte à outils de MétéoLocale (modules 07 à 09)
import csv

VILLES = ["Toulouse", "Montpellier", "Nîmes", "Perpignan", "Albi", "Tarbes"]

def moyenne(valeurs):
    '''Moyenne d'une liste non vide de nombres.'''
    return ...

def celsius_vers_fahrenheit(celsius, chiffres=1):
    '''Convertit des °C en °F, arrondi à chiffres chiffres.'''
    return ...

def charger_meteo(chemin):
    '''Lit un CSV MétéoLocale propre et renvoie la liste des relevés convertis.'''
    releves = []
    with open(chemin, encoding="utf-8", newline="") as fichier:
        for ligne in csv.DictReader(fichier):
            releves.append({
                "date": ligne["date"], "ville": ligne["ville"],
                "temp_min": float(ligne["temp_min"]), "temp_max": float(ligne["temp_max"]),
                "pluie_mm": float(ligne["pluie_mm"]), "source": "historique",
            })
    return releves
"""
with open("sorties/meteolocale.py", "w", encoding="utf-8") as fichier:
    fichier.write(code)

from sorties import meteolocale
importlib.reload(meteolocale)
print("Module meteolocale prêt :", len(meteolocale.VILLES), "villes")

In [ ]:
# Vérification
assert meteolocale.moyenne([10, 20]) == 15.0, f"meteolocale.moyenne([10, 20]) doit valoir 15.0, elle renvoie {meteolocale.moyenne([10, 20])!r} : complète la formule dans le texte code."
assert meteolocale.celsius_vers_fahrenheit(25) == 77.0, f"meteolocale.celsius_vers_fahrenheit(25) doit valoir 77.0, elle renvoie {meteolocale.celsius_vers_fahrenheit(25)!r}."
print("✅ OK")

### Étape 2 — L'historique de Toulouse

**Objectif** : Utiliser ton module pour charger l'historique.

**À faire** :
1. Charge `../donnees/meteo.csv` avec `meteolocale.charger_meteo(...)`.
2. Range dans `historique` les relevés de Toulouse seulement.

**Résultat attendu** :
```
30 jours d'historique, du 2026-09-01 au 2026-09-30
```

<details><summary>💡 Indice</summary>

Une compréhension de liste avec `if r["ville"] == "Toulouse"` (module 05), ou une boucle avec `append`.

📖 Cours, section 10.

</details>

In [ ]:
tous = meteolocale.charger_meteo("../donnees/meteo.csv")
historique = ...
print(len(historique), "jours d'historique, du", historique[0]["date"], "au", historique[-1]["date"])

In [ ]:
# Vérification
assert "historique" in dir() and type(historique) == list, "historique doit être une liste de relevés."
assert len(historique) == 30, f"historique doit contenir 30 relevés de Toulouse, tu en as {len(historique)}."
assert historique[0]["temp_max"] == 25.9 and historique[0]["source"] == "historique", "Les relevés doivent venir de meteolocale.charger_meteo (nombres convertis, source 'historique')."
print("✅ OK")

### Étape 3 — Les prévisions d'octobre

**Objectif** : Lire la réponse de l'API et la transformer en relevés de même forme que l'historique.

**À faire** :
1. Écris `lire_previsions(chemin, ville)` : elle lit le JSON (`json.load`) et renvoie la liste des relevés (`date`, `ville`, `temp_min`, `temp_max`, `pluie_mm`, `source` = `"prévision"`).
2. Range dans `previsions` le résultat pour `../donnees/open_meteo_toulouse.json` et `"Toulouse"`.

**Résultat attendu** :
```
7 prévisions
{'date': '2026-10-01', 'ville': 'Toulouse', 'temp_min': 11.2, 'temp_max': 22.1, 'pluie_mm': 0.0, 'source': 'prévision'}
```

<details><summary>💡 Indice</summary>

Une boucle sur `range(len(quotidien["time"]))` qui ajoute un dictionnaire par position : c'est la fonction `previsions_en_releves` du cours.

📖 Cours, section 10.

</details>

In [ ]:
def lire_previsions(chemin, ville):
    """Réponse Open-Meteo enregistrée -> liste de relevés (un par jour)."""
    with open(chemin, encoding="utf-8") as fichier:
        quotidien = json.load(fichier)["daily"]
    releves = []
    ...
    return releves

previsions = lire_previsions("../donnees/open_meteo_toulouse.json", "Toulouse")
print(len(previsions), "prévisions")
print(previsions[0])

In [ ]:
# Vérification
assert len(previsions) == 7, f"previsions doit contenir 7 jours, tu en as {len(previsions)}."
attendu = {"date": "2026-10-01", "ville": "Toulouse", "temp_min": 11.2, "temp_max": 22.1, "pluie_mm": 0.0, "source": "prévision"}
assert previsions[0] == attendu, f"1re prévision inattendue :\n{previsions[0]}\nattendu :\n{attendu}"
print("✅ OK")

### Étape 4 — La fusion

**Objectif** : Mettre l'historique et les prévisions bout à bout et ajouter le nom du jour.

**À faire** :
1. Range dans `toulouse` l'historique suivi des prévisions.
2. Ajoute à **chaque** relevé de `toulouse` une clé `"jour"` : le nom du jour en français (avec `JOURS` et `date.fromisoformat`).

**Résultat attendu** :
```
37 jours : du mardi 2026-09-01 au mercredi 2026-10-07
```

<details><summary>💡 Indice</summary>

`historique + previsions`. Dans la boucle : `JOURS[date.fromisoformat(releve["date"]).weekday()]`.

📖 Cours, sections 4 et 10.

</details>

In [ ]:
toulouse = ...
for releve in toulouse:
    releve["jour"] = ...

premier, dernier = toulouse[0], toulouse[-1]
print(len(toulouse), "jours : du", premier["jour"], premier["date"], "au", dernier["jour"], dernier["date"])

In [ ]:
# Vérification
assert type(toulouse) == list and len(toulouse) == 37, "toulouse doit contenir 30 + 7 = 37 relevés : historique + previsions."
assert toulouse[0]["jour"] == "mardi", f"Le 2026-09-01 est un mardi, tu as {toulouse[0].get('jour')!r}."
assert toulouse[-1]["jour"] == "mercredi", f"Le 2026-10-07 est un mercredi, tu as {toulouse[-1].get('jour')!r}."
print("✅ OK")

### Étape 5 — Septembre contre octobre

**Objectif** : Comparer les deux périodes avec `statistics` et ton module.

**À faire** :
1. Range dans `moy_sept` et `moy_oct` les moyennes des `temp_max` (historique / prévisions), avec `statistics.mean`, arrondies à 1 chiffre.
2. Range dans `difference` l'écart `moy_oct - moy_sept` arrondi à 1 chiffre, et dans `moy_oct_f` la moyenne d'octobre en °F (avec ton module).

**Résultat attendu** :
```
Septembre 23.9 °C, début octobre 21.0 °C (69.8 °F) : -2.9 °C
```

<details><summary>💡 Indice</summary>

`statistics.mean([r["temp_max"] for r in historique])`, puis `round(..., 1)`. Pour les °F : `meteolocale.celsius_vers_fahrenheit(moy_oct)`.

📖 Cours, section 5.

</details>

In [ ]:
moy_sept = ...
moy_oct = ...
difference = ...
moy_oct_f = ...
print(f"Septembre {moy_sept} °C, début octobre {moy_oct} °C ({moy_oct_f} °F) : {difference} °C")

In [ ]:
# Vérification
assert moy_sept == 23.9, f"moy_sept doit valoir 23.9, tu as {moy_sept!r}."
assert moy_oct == 21.0, f"moy_oct doit valoir 21.0, tu as {moy_oct!r}."
assert difference == -2.9, f"difference doit valoir -2.9 (octobre plus frais), tu as {difference!r} : moy_oct - moy_sept."
assert moy_oct_f == 69.8, f"moy_oct_f doit valoir 69.8, tu as {moy_oct_f!r} : meteolocale.celsius_vers_fahrenheit(moy_oct)."
print("✅ OK")

### Étape 6 — Le fichier pour le site

**Objectif** : Écrire la fusion en JSON et produire le bandeau des prévisions.

**À faire** :
1. Écris le dictionnaire `export = {"ville": "Toulouse", "moyenne_septembre": moy_sept, "releves": toulouse}` dans `sorties/toulouse_site.json` (`ensure_ascii=False, indent=2`).
2. Construis la liste `bandeau` : pour chaque prévision, `"jeudi 1/10 : 22.1 °C"` (jour, numéro du jour, `/`, numéro du mois, temp_max). Utilise `date.fromisoformat` pour `.day` et `.month`.

**Résultat attendu** :
```
jeudi 1/10 : 22.1 °C
vendredi 2/10 : 21.4 °C
samedi 3/10 : 19.8 °C
dimanche 4/10 : 18.2 °C
lundi 5/10 : 20.5 °C
mardi 6/10 : 23.0 °C
mercredi 7/10 : 21.7 °C
```

<details><summary>💡 Indice</summary>

`json.dump(export, fichier, ensure_ascii=False, indent=2)`. Pour la ligne : `f"{r['jour']} {d.day}/{d.month} : {r['temp_max']} °C"` (la clé `jour` a été ajoutée à l'étape 4).

📖 Cours, sections 4 et 10 ; module 08, section 8.

</details>

In [ ]:
export = {"ville": "Toulouse", "moyenne_septembre": moy_sept, "releves": toulouse}
with open("sorties/toulouse_site.json", "w", encoding="utf-8") as fichier:
    ...

bandeau = []
for r in previsions:
    d = date.fromisoformat(r["date"])
    bandeau.append(...)

for ligne in bandeau:
    print(ligne)

In [ ]:
# Vérification
with open("sorties/toulouse_site.json", encoding="utf-8") as f:
    texte = f.read()
assert texte != "", "Le fichier sorties/toulouse_site.json est vide : appelle json.dump(export, fichier, ensure_ascii=False, indent=2)."
with open("sorties/toulouse_site.json", encoding="utf-8") as f:
    relu = json.load(f)
assert len(relu["releves"]) == 37 and relu["moyenne_septembre"] == 23.9, "Le JSON doit contenir les 37 relevés et la moyenne de septembre."
assert bandeau[0] == "jeudi 1/10 : 22.1 °C", f"1re ligne du bandeau inattendue : {bandeau[0]!r}"
assert bandeau[-1] == "mercredi 7/10 : 21.7 °C", f"Dernière ligne du bandeau inattendue : {bandeau[-1]!r}"
print("✅ OK")

## 🏁 Ce que tu as construit

Ton **propre module** `sorties/meteolocale.py`, réutilisable partout ; l'historique de septembre et les prévisions
d'octobre **fusionnés** dans une même liste, avec le jour de la semaine ; une comparaison chiffrée
(octobre démarre près de 3 °C plus frais) ; et `sorties/toulouse_site.json`, prêt pour le site de Karim.

**Au module 10**, tu découvriras **pandas** et **matplotlib** : tout ce que tu as fait « à la main » depuis le
module 05 (lire, filtrer, regrouper par ville, faire des moyennes) tiendra en quelques lignes... et tu dessineras
tes premiers graphiques pour le bulletin mensuel.